# Simulating quantum volume circuits

This notebook demonstrates how to use quax to simulate the a common class of random quantum circuits for quantum volume experiments.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import jax
import jax.numpy as jnp

import quax as qx

## Parameters

The notebook parameters are below

In [ ]:
num_qubits = 6
num_layers = num_qubits  # QV circuits have depth = width
depolarizing_prob = 0.02  # per-gate depolarizing probability
n_trajectories = 10_000
seed = 42

## Create the circuit

Quantum volume circuits are created by applying layers of random 2-qubit unitaries. In each layer, the qubits are randomly paired and a Haar-random SU(4) unitary is applied to each pair.

Below, we construct the random unitaries and the qubit indices they act on. We then use `qx.targeted_apply_unitary` to compute the ideal (noiseless) final state of the circuit.

In [ ]:
key = jax.random.key(seed)

# Generate random qubit pairings and unitaries for each layer
circuit = []
for layer in range(num_layers):
    key, perm_key = jax.random.split(key)
    # Random permutation determines qubit pairing
    perm = jax.random.permutation(perm_key, num_qubits)
    pairs = [(int(perm[i]), int(perm[i + 1])) for i in range(0, num_qubits, 2)]

    layer_gates = []
    for pair in pairs:
        key, gate_key = jax.random.split(key)
        U = qx.random_unitary(dims=((2, 2), (2, 2)), key=gate_key)
        layer_gates.append((U, pair))
    circuit.append(layer_gates)

# Apply the circuit to |0...0⟩
psi_ideal = qx.zero_state_vector(num_qubits)
for layer_gates in circuit:
    for U, pair in layer_gates:
        psi_ideal = qx.targeted_apply_unitary(U, psi_ideal, pair)

print(f"Circuit: {num_layers} layers, {num_qubits} qubits")
print(f"Ideal state vector shape: {psi_ideal.matrix.shape}")

### Calculate HOP

We can calculate the Heavy Output Probability for the circuit.

In [ ]:
# Compute ideal output probabilities
ideal_probs = jnp.abs(psi_ideal.matrix) ** 2

# Heavy outputs are those with probability above the median
median_prob = jnp.median(ideal_probs)
heavy_mask = ideal_probs > median_prob

# HOP = probability of sampling a heavy output
hop_ideal = jnp.sum(ideal_probs[heavy_mask])
print(f"Ideal HOP: {hop_ideal:.4f}")
print("(QV threshold is HOP > 2/3 ≈ 0.6667)")

### Add some noise

Real quantum gates aren't perfect. We'll add some noise to each gate, and calculate the final state.

In [ ]:
# Create a 2-qubit depolarizing Kraus map
s_2q = qx.channels.depolarizing(jnp.array(depolarizing_prob), dims=(2, 2))
kraus_2q = qx.superop_to_kraus(s_2q)

# Start from |0...0⟩ as a density matrix and apply each gate + noise
rho_noisy = qx.zero_state_matrix(num_qubits)
for layer_gates in circuit:
    for U, pair in layer_gates:
        # Apply the ideal gate
        rho_noisy = qx.targeted_apply_superop(qx.unitary_to_superop(U), rho_noisy, pair)
        # Apply depolarizing noise after each gate
        rho_noisy = qx.targeted_apply_kraus_map(kraus_2q, rho_noisy, pair)

print(f"Noisy density matrix shape: {rho_noisy.matrix.shape}")

## Calculate the noisy HOP

We'll calculate the noisy HOP for comparison

In [ ]:
# Noisy probabilities from the diagonal of the density matrix
noisy_probs = jnp.real(jnp.diag(rho_noisy.matrix))

# HOP using the same heavy outputs defined by the ideal circuit
hop_noisy_dm = jnp.sum(noisy_probs[heavy_mask])
print(f"Noisy HOP (density matrix): {hop_noisy_dm:.4f}")
print(f"Ideal HOP:                  {hop_ideal:.4f}")
print(f"HOP degradation:            {hop_ideal - hop_noisy_dm:.4f}")

## Trajectory simulation

Our density matrix approach scales well to about a dozen qubits. But if we want to simulate noisy circuits larger than that, we'll need to use the state trajectory approach.

Let's use 10,000 trajectories and see that we get the same HOP as the density matrix simulation.

In [ ]:
# Generate an ensemble of keys — one per trajectory
key, traj_key = jax.random.split(key)


@jax.jit
def simulate_trajectories(psi, traj_key):
    """Run the circuit with depolarizing noise using trajectory sampling."""
    for layer_gates in circuit:
        for U, pair in layer_gates:
            psi = qx.targeted_apply_unitary(U, psi, pair)
            traj_key, noise_key = jax.random.split(traj_key)
            sample_keys = jax.random.split(noise_key, num=n_trajectories)
            psi = qx.targeted_apply_kraus_map_trajectory(kraus_2q, psi, sample_keys, pair)
    return psi


psi_traj = simulate_trajectories(qx.zero_state_vector(num_qubits), traj_key)

# Compute per-trajectory probabilities and average
traj_probs = jnp.mean(jnp.abs(psi_traj.matrix) ** 2, axis=0)

# HOP from trajectory simulation
hop_trajectory = jnp.sum(traj_probs[heavy_mask])
print(f"Trajectory HOP ({n_trajectories:,} shots): {hop_trajectory:.4f}")
print(f"Density matrix HOP:                  {hop_noisy_dm:.4f}")
print(f"Difference:                          {abs(hop_trajectory - hop_noisy_dm):.4f}")